[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somanshusingla/llm-from-scratch/blob/main/notebooks/04_ch5_pretraining.ipynb)

# Chapter 5 — Pretraining on Unlabeled Data

**Build a Large Language Model (From Scratch)** · notebook 4 of 7

In Chapter 4 the model produced gibberish because its weights were random. Now we
**train** it. You'll:

1. Measure how "wrong" the model is with a **loss** (cross-entropy).
2. Write the **training loop** and watch loss drop + text become coherent.
3. Control generation with **temperature** and **top-k** sampling.
4. **Save and load** model weights.
5. Load **OpenAI's pretrained GPT-2 weights** into *your* from-scratch model and
   generate genuinely fluent text. 🎉

> **CPU note:** training the full 124M model from scratch needs a GPU. So for the
> from-scratch demo we train a **small** model (a few million params) on one short
> story — it finishes in ~1-2 minutes on a laptop CPU and clearly shows learning.
> The final section loads the *real* 124M GPT-2, which runs fine on CPU for
> inference. On Colab with a GPU you can bump the config up to the full 124M.

## 0. Setup

In [ ]:
import importlib, subprocess, sys
def ensure(pkg, name=None):
    try: importlib.import_module(name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
for p, n in [("tiktoken", None), ("matplotlib", None)]:
    ensure(p, n)

import torch, torch.nn as nn, tiktoken, os, urllib.request
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", device)

## Recap: the GPT model (from Chapter 4)

One cell with all the pieces so this notebook is self-contained. Run it and move
on — it's the same code you already understand.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out, self.num_heads, self.head_dim = d_out, num_heads, d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1))
    def forward(self, x):
        b, n, _ = x.shape
        k = self.W_key(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        q = self.W_query(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_value(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(2, 3)
        scores.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        w = torch.softmax(scores / k.shape[-1]**0.5, dim=-1)
        w = self.dropout(w)
        ctx = (w @ v).transpose(1, 2).contiguous().view(b, n, self.d_out)
        return self.out_proj(ctx)

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]))
    def forward(self, x): return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"],
            cfg["context_length"], cfg["drop_rate"], cfg["n_heads"], cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        x = x + self.drop_shortcut(self.att(self.norm1(x)))
        x = x + self.drop_shortcut(self.ff(self.norm2(x)))
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
    def forward(self, in_idx):
        b, seq_len = in_idx.shape
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        return self.out_head(self.final_norm(x))

def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)[:, -1, :]
        idx_next = torch.argmax(torch.softmax(logits, dim=-1), dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

def text_to_token_ids(text, tokenizer):
    return torch.tensor(tokenizer.encode(text, allowed_special={"<|endoftext|>"})).unsqueeze(0)
def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

tokenizer = tiktoken.get_encoding("gpt2")
print("Model code ready.")

## 5.1 Evaluating generative text models

**Loss** measures how surprised the model is by the correct next token. We use
**cross-entropy**: lower is better. A random model over 50,257 tokens starts near
`ln(50257) ≈ 10.8`.

First, the data. We reuse *"The Verdict"* and build train/validation dataloaders
with the sliding-window `Dataset` from Chapter 2.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids, self.target_ids = [], []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i:i+max_length]))
            self.target_ids.append(torch.tensor(token_ids[i+1:i+max_length+1]))
    def __len__(self): return len(self.input_ids)
    def __getitem__(self, idx): return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size, max_length, stride, shuffle, drop_last, num_workers=0):
    ds = GPTDatasetV1(txt, tiktoken.get_encoding("gpt2"), max_length, stride)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=num_workers)

file_path = "the-verdict.txt"
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/"
       "ch02/01_main-chapter-code/the-verdict.txt")
if not os.path.exists(file_path):
    try:
        urllib.request.urlretrieve(url, file_path)
    except Exception as e:
        print("Download failed:", e, "-- using fallback sample.")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(("I HAD always thought Jack Gisburn rather a cheap genius--though a "
                     "good fellow enough--so it was no great surprise to me to hear that "
                     "he had dropped his painting and married a rich widow. ") * 60)
with open(file_path, "r", encoding="utf-8") as f:
    text_data = f.read()
print("Characters:", len(text_data), "| BPE tokens:", len(tokenizer.encode(text_data)))

### The training configuration (small, for CPU)

We use a compact model so training finishes quickly on a CPU while still clearly
demonstrating learning. (The book uses the full 124M config — try that on a Colab
GPU by swapping in the commented values.)

In [ ]:
# The config auto-scales: full GPT-2 124M architecture on a GPU, a small model on CPU.
if device.type == "cuda":
    GPT_CONFIG = {
        "vocab_size": 50257, "context_length": 256,
        "emb_dim": 768, "n_heads": 12, "n_layers": 12,   # full GPT-2 124M width/depth
        "drop_rate": 0.1, "qkv_bias": False,
    }
    NUM_EPOCHS, BATCH_SIZE = 30, 8      # more epochs on GPU
else:
    GPT_CONFIG = {
        "vocab_size": 50257, "context_length": 256,
        "emb_dim": 384, "n_heads": 6, "n_layers": 6,     # small model for CPU
        "drop_rate": 0.1, "qkv_bias": False,
    }
    NUM_EPOCHS, BATCH_SIZE = 10, 2
print("Training config:", GPT_CONFIG)
print("Epochs:", NUM_EPOCHS, "| batch size:", BATCH_SIZE)

# train/validation split
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data, val_data = text_data[:split_idx], text_data[split_idx:]

train_loader = create_dataloader_v1(
    train_data, batch_size=BATCH_SIZE, max_length=GPT_CONFIG["context_length"],
    stride=GPT_CONFIG["context_length"], shuffle=True, drop_last=True)
val_loader = create_dataloader_v1(
    val_data, batch_size=BATCH_SIZE, max_length=GPT_CONFIG["context_length"],
    stride=GPT_CONFIG["context_length"], shuffle=False, drop_last=False)

print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    return torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    num_batches = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))
    for i, (inp, tgt) in enumerate(data_loader):
        if i >= num_batches: break
        total_loss += calc_loss_batch(inp, tgt, model, device).item()
    return total_loss / num_batches

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG).to(device)
print("Model params:", f"{sum(p.numel() for p in model.parameters()):,}")

with torch.no_grad():
    print("Initial train loss:", round(calc_loss_loader(train_loader, model, device), 3))
    print("Initial val loss:  ", round(calc_loss_loader(val_loader, model, device), 3))

## 5.2 Training an LLM

The loop is the standard deep-learning recipe: for each batch, compute the loss,
backpropagate, and let the AdamW optimizer nudge the weights. Every few steps we
print train/val loss and a text sample so you can watch progress.

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(model, encoded, 50, context_size)
    print("   sample:", token_ids_to_text(token_ids, tokenizer).replace("\n", " "))
    model.train()

def train_model_simple(model, train_loader, val_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1
            if global_step % eval_freq == 0:
                tr, vl = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(tr); val_losses.append(vl); track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (step {global_step:04d}): train {tr:.3f}, val {vl:.3f}")
        generate_and_print_sample(model, tokenizer, device, start_context)
    return train_losses, val_losses, track_tokens_seen

In [ ]:
import time
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)

t0 = time.time()
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=NUM_EPOCHS, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer)
print(f"\nTraining took {time.time()-t0:.1f}s on {device}")

Watch the sample text at the end of each epoch: it starts as noise and gradually
turns into phrases from the story. The training loss keeps dropping; the
validation loss flattens and rises — classic **overfitting** on a tiny dataset.
That's fine here: our goal is to see the mechanics of learning, not to build a
general model. Let's plot it.

In [ ]:
def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(6, 4))
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Steps (evaluated)"); ax1.set_ylabel("Loss"); ax1.legend()
    plt.title("Training vs validation loss"); plt.tight_layout(); plt.show()

steps = list(range(len(train_losses)))
plot_losses(steps, tokens_seen, train_losses, val_losses)

## 5.3 Decoding strategies to control randomness

Greedy decoding (always pick the most likely token) is repetitive. Two knobs add
diversity:

- **Temperature** — divide logits by T before softmax. `T<1` = more focused,
  `T>1` = more random, `T→0` = greedy.
- **Top-k** — only sample among the k most likely tokens, ignoring the long tail.

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)[:, -1, :]
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val,
                                 torch.tensor(float("-inf")).to(logits.device), logits)
        if temperature > 0.0:
            probs = torch.softmax(logits / temperature, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if eos_id is not None and (idx_next == eos_id).all():
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

torch.manual_seed(123)
out = generate(model, text_to_token_ids("Every effort moves you", tokenizer).to(device),
               max_new_tokens=25, context_size=GPT_CONFIG["context_length"],
               temperature=1.2, top_k=25)
print(token_ids_to_text(out, tokenizer))

## 5.4 Loading and saving model weights in PyTorch

Never lose your training progress. Save the model's `state_dict` (and the
optimizer's, if you want to resume training).

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, "model_and_optimizer.pth")
print("Saved to model_and_optimizer.pth")

# reload into a fresh model
checkpoint = torch.load("model_and_optimizer.pth", map_location=device, weights_only=True)
model2 = GPTModel(GPT_CONFIG).to(device)
model2.load_state_dict(checkpoint["model_state_dict"])
model2.eval()
print("Reloaded model; val loss:", round(calc_loss_loader(val_loader, model2, device), 3))

## 5.5 Loading pretrained weights from OpenAI 🎉

Training a good model from scratch needs huge data + compute. Instead, we load
OpenAI's **pretrained GPT-2 (124M)** weights — obtained here via the
`transformers` library — directly into *our* from-scratch `GPTModel`. Same
architecture, real weights.

This is the payoff: your hand-built model will now write fluent English.

> Downloads ~500 MB the first time (cached afterwards). Needs internet. If it
> fails, the cell prints a note and the rest of the notebook still works.

In [ ]:
# GPT-2 124M config -- note qkv_bias=True (GPT-2 uses biases in attention)
GPT2_124M = {
    "vocab_size": 50257, "context_length": 1024, "emb_dim": 768,
    "n_heads": 12, "n_layers": 12, "drop_rate": 0.0, "qkv_bias": True,
}

def _copy(dst, src):
    assert dst.shape == src.shape, f"shape mismatch {tuple(dst.shape)} vs {tuple(src.shape)}"
    with torch.no_grad():
        dst.copy_(src)

def load_weights_from_hf(gpt, hf_model):
    sd = hf_model.state_dict()
    _copy(gpt.tok_emb.weight, sd["transformer.wte.weight"])
    _copy(gpt.pos_emb.weight, sd["transformer.wpe.weight"])
    for b in range(len(gpt.trf_blocks)):
        p = f"transformer.h.{b}."
        blk = gpt.trf_blocks[b]
        # combined QKV weight (Conv1D stores as [in, out] -> transpose for Linear)
        w = sd[p + "attn.c_attn.weight"]
        q_w, k_w, v_w = w.split(w.shape[1] // 3, dim=1)
        _copy(blk.att.W_query.weight, q_w.T)
        _copy(blk.att.W_key.weight,   k_w.T)
        _copy(blk.att.W_value.weight, v_w.T)
        bqkv = sd[p + "attn.c_attn.bias"]
        q_b, k_b, v_b = bqkv.split(bqkv.shape[0] // 3, dim=0)
        _copy(blk.att.W_query.bias, q_b)
        _copy(blk.att.W_key.bias,   k_b)
        _copy(blk.att.W_value.bias, v_b)
        _copy(blk.att.out_proj.weight, sd[p + "attn.c_proj.weight"].T)
        _copy(blk.att.out_proj.bias,   sd[p + "attn.c_proj.bias"])
        _copy(blk.norm1.scale, sd[p + "ln_1.weight"]); _copy(blk.norm1.shift, sd[p + "ln_1.bias"])
        _copy(blk.norm2.scale, sd[p + "ln_2.weight"]); _copy(blk.norm2.shift, sd[p + "ln_2.bias"])
        _copy(blk.ff.layers[0].weight, sd[p + "mlp.c_fc.weight"].T)
        _copy(blk.ff.layers[0].bias,   sd[p + "mlp.c_fc.bias"])
        _copy(blk.ff.layers[2].weight, sd[p + "mlp.c_proj.weight"].T)
        _copy(blk.ff.layers[2].bias,   sd[p + "mlp.c_proj.bias"])
    _copy(gpt.final_norm.scale, sd["transformer.ln_f.weight"])
    _copy(gpt.final_norm.shift, sd["transformer.ln_f.bias"])
    _copy(gpt.out_head.weight, sd["transformer.wte.weight"])  # weight tying

In [ ]:
import os, warnings
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore")

gpt2 = None
try:
    from transformers import GPT2LMHeadModel
    try:
        from transformers.utils import logging as hf_logging
        hf_logging.disable_progress_bar(); hf_logging.set_verbosity_error()
    except Exception:
        pass
    print("Downloading / loading OpenAI GPT-2 124M (first time ~500MB)...")
    hf_model = GPT2LMHeadModel.from_pretrained("gpt2")
    hf_model.eval()

    gpt2 = GPTModel(GPT2_124M)
    load_weights_from_hf(gpt2, hf_model)
    gpt2.to(device).eval()
    print("Loaded pretrained weights into our from-scratch GPTModel.")
except Exception as e:
    print("Could not load pretrained GPT-2:", repr(e))
    print("Skipping this section -- the rest of the notebook already ran fine.")

In [ ]:
if gpt2 is not None:
    torch.manual_seed(123)
    out = generate(
        model=gpt2,
        idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
        max_new_tokens=40, context_size=GPT2_124M["context_length"],
        temperature=0.7, top_k=40,
    )
    print(token_ids_to_text(out, tokenizer))
else:
    print("(pretrained model unavailable -- see the note above)")

Real, fluent English — from a model *you* built. The architecture in this
notebook is identical to OpenAI's GPT-2; only the learned weights differ.

## Summary

- **Loss** (cross-entropy) quantifies prediction quality; the **training loop**
  minimizes it with backprop + AdamW.
- **Temperature** and **top-k** trade off between safe and creative generation.
- `state_dict` save/load lets you checkpoint and resume.
- The same architecture accepts **OpenAI's pretrained weights** and produces
  fluent text.

### Try it yourself
- On a Colab GPU, set `GPT_CONFIG` to the full 124M values and train longer.
- Compare `generate(..., temperature=0.2)` vs `temperature=1.5`.

**Next:** `05_ch6_classification_finetuning.ipynb` — fine-tune GPT-2 into a spam
classifier.